# Longor M3 launcher (`libero_10` task 0)

Runs the full Longor path (planner + executor + VLA service) on `libero_10` task 0.

Goal for M3:
- At least one successful end-to-end episode (`success=True` in run summary).

Notes:
- Uses current runtime default `VLA_ACTIONS_PER_CHUNK` from `src/config.py`.
- Section 7 streams full stdout/stderr live from a single run.


## 1 — Clone repo and capture secrets

In [1]:
import getpass

github_token = getpass.getpass('GitHub PAT (repo scope): ')
openai_api_key = getpass.getpass('OpenAI API key: ')

In [2]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 — Headless rendering + app deps

In [3]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt
%pip install -e .

## 3 — SmolVLA venv

In [4]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    subprocess.run([VLA_PYTHON, '-m', 'pip', *args], check=True)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
vla_pip('install', '-U', 'pip')
vla_pip('install', '-r', 'deps/requirements-vla-smolvla-linux.txt')
vla_pip('install', '-e', '.')

## 4 — Configure `.env` for Longor run

In [5]:
with open('.env', 'w', encoding='utf-8') as f:
    f.write('\n'.join([
        f'OPENAI_API_KEY={openai_api_key}',
        'LOG_LEVEL=INFO',
        f'LLM_MODEL=gpt-4.1-mini',
        'SAVE_VIDEO=0',
        'LOG_LEVEL=DEBUG',
    ]) + '\n')

print('.env written')

## 5 — Start VLA server + health

In [6]:
import os, subprocess, time, urllib.request

VLA_LOG = '/tmp/longor_m3_vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
log = open(VLA_LOG, 'w', buffering=1)
proc = subprocess.Popen(
    [vla_python, 'services/smolvla_service.py'],
    cwd='/content/Longor',
    stdout=log,
    stderr=subprocess.STDOUT,
    env=dict(os.environ),
)

health_ok = False
for elapsed in range(0, 240, 5):
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('Health:', resp.read().decode())
        health_ok = True
        break
    except Exception:
        print(f'Waiting for /health ({elapsed}s)...')
        time.sleep(5)

if not health_ok:
    raise RuntimeError(f'VLA server not healthy; see {VLA_LOG}')

## 6 — Sanity check action-chunk setting

In [7]:
# the notebook is just a launcher; running "import config" directly will fail.
!python -c "import config; print('VLA_ACTIONS_PER_CHUNK =', config.VLA_ACTIONS_PER_CHUNK)"

## 7 — Run Longor once (`libero_10`, task 0)


In [8]:
!python examples/long_horizon_llm_libero.py

In [16]:
!python validation/libero_probe.py --suite libero_10 --task 0 --episodes 1

In [22]:
!python validation/libero_probe.py --suite libero_object --task 0 --episodes 1